# VL-JEPA: Real Models & Data

**No mocks - loads actual V-JEPA2, Llama, Gemma, and MSR-VTT**

Requires: `huggingface-cli login` for Llama access

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer, AutoImageProcessor, LlamaForCausalLM
from datasets import load_dataset
from PIL import Image
import torchvision.transforms as transforms
from tqdm import tqdm
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 1. Load Vision Encoder (V-JEPA2 or ViT-Large)

In [ ]:
# Load V-JEPA2 or ViT-L from HuggingFace
try:
    vision_model = AutoModel.from_pretrained('facebook/vjepa-vitl-16', torch_dtype=torch.float16, device_map='auto')
    vision_processor = AutoImageProcessor.from_pretrained('facebook/vjepa-vitl-16')
    print('✓ Loaded V-JEPA2')
except:
    vision_model = AutoModel.from_pretrained('google/vit-large-patch16-224', torch_dtype=torch.float16, device_map='auto')
    vision_processor = AutoImageProcessor.from_pretrained('google/vit-large-patch16-224')
    print('✓ Loaded ViT-Large-16')

for param in vision_model.parameters():
    param.requires_grad = False
vision_model.eval()
print(f'Vision params: {sum(p.numel() for p in vision_model.parameters())/1e6:.1f}M (frozen)')